# Hypothesis 2 — Main Notebook

Hypothesis: The underlying LLM of an MAAI statistically significantly affects its fairness judgments; specifically, American and Chinese LLMs differ in their judgments.

This notebook prepares two groups of aligned experiment configurations (American, Chinese),
runs them selectively in parallel, and compares the outcome distributions across groups (5 categories).

- 34 configs per group (default), total 68.
- Per-config: a shared temperature drawn from U(0, 1.5), shared random seed, and agent models selected per group.
- Language is English for all runs.
- Utility agent is `gemini-2.5-flash`.
- Voting detection mode is set to "complex" for all runs.
- Exact test: Fisher–Freeman–Halton via R (if available). No Chi-square fallback.
- Effect size: Cramér's V (bias-corrected, as in Hypothesis 1).


# 1. Imports

In [1]:
import pandas as pd
import numpy as np
import sys, os
from pathlib import Path

# Ensure repo root on sys.path (for local package imports)
def _add_repo_root_to_sys_path():
    here = Path.cwd().resolve()
    for p in [here] + list(here.parents):
        if (p / 'main.py').exists() and (p / 'hypothesis_testing').is_dir():
            if str(p) not in sys.path:
                sys.path.insert(0, str(p))
            return p
    return here
_REPO_ROOT = _add_repo_root_to_sys_path()

import json
import random
import shutil
import yaml
from collections import Counter
from hypothesis_testing.utils_hypothesis_testing.runner import (
    list_config_files,
    select_configs,
    run_configs_in_parallel,
)


# 2. Model Selection Process

## Data Import & Cleaning

In [2]:
# Read the CSV file into a pandas dataframe
# Data scraped from https://artificialanalysis.ai/leaderboards/models on 2025-10-08
df = pd.read_csv("model_overview_artificial_analysis.csv")

In [ ]:
# Remove '€' symbol and spaces, replace comma with dot, and convert to float
df['Price (Blended EUR/1M Tokens)'] = df['Price (Blended EUR/1M Tokens)'].str.replace('€', '').str.strip().str.replace(',', '.').astype(float)

# Convert Context Window (handle 'k' and 'm', allow commas/decimals)
s = df['Context Window'].astype(str).str.strip().str.lower()
m = s.str.extract(r'^\s*(\d+(?:[.,]\d+)?)\s*([km]?)\s*$')  # number + optional suffix
num = m[0].str.replace(',', '.', regex=False).astype(float)
mult = m[1].map({'k': 1_000, 'm': 1_000_000}).fillna(1)
df['Context Window'] = num * mult

# Convert Latency First Answer Chunk (s) (replace comma with dot)
df['Latency First Answer Chunk (s)'] = (
    df['Latency First Answer Chunk (s)']
      .astype(str)
      .str.replace(',', '.', regex=False)
      .astype(float)
)

In [12]:
# Filter for models with price less than 0.5 EUR per 1M tokens to reduce cost
df_cheap = df[df['Price (Blended EUR/1M Tokens)'] < 0.5]

TypeError: '<' not supported between instances of 'str' and 'float'

In [ ]:
# Sort by Intelligence Index (Artificial Analysis Index)
df_cheap= df_cheap.sort_values(by=["Intelligence Index"], ascending=False)

## Selecting the Chinese Models

In [ ]:
# Selecting the Chinese Models for the analysis

# Filtering by Chinese models
df_chinese = df_cheap[df_cheap["Country"] == "China"]

# Removing models which are not properly accesible through API
df_chinese = df_chinese[df_chinese["Speed (Median Tokens/s)"]>0]

#Selecting 6 best models by Articialy Intelligence Index
df_chinese_select = df_chinese[:6]



# List of Chinese models
chinese_models = df_chinese_select["Model"].tolist()

df_chinese_select.head()

NameError: name 'df_cheap' is not defined

## Selecting the American Models

In [ ]:
df_american = df_cheap[df_cheap["Country"] == "USA"] 

df_american = df_american[df_american["Speed (Median Tokens/s)"]>0]

# Exclude GPT 5 models since they do not support temperature
df_american = df_american[~df_american["Model"].str.contains(r"gpt[-\s]*5", case=False, na=False)]

df_american_select = df_american[:6]


df_american_select.head()

# List of selected models
us_models = df_american_select["Model"].tolist()

NameError: name 'df_cheap' is not defined

In [ ]:
print(chinese_models)
print(us_models)


['DeepSeek V3.2 Exp', 'GLM-4.5-Air', 'DeepSeek V3.2 Exp', 'Qwen3 Omni 30B A3B', 'QwQ-32B', 'Qwen3 30B A3B 2507']
['Grok 4 Fast', 'gpt-oss-120B (high)', 'Grok Code Fast 1', 'Gemini 2.5 Flash-Lite (Sep)', 'Llama Nemotron Super 49B v1.5', 'gpt-oss-20B (high)']


## Retrieval of OpenRouter IDs

In [ ]:
model_lookup = {
"DeepSeek V3.2 Exp": "deepseek/deepseek-v3.2-exp",
"GLM-4.5-Air": "z-ai/glm-4.5-air",
"Qwen3 Omni 30B A3B": "qwen/qwen3-next-80b-a3b-thinking",
"QwQ-32B": "qwen/qwq-32b",
"Qwen3 30B A3B 2507": "qwen/qwen3-30b-a3b-instruct-2507",
"Grok 4 Fast": "x-ai/grok-4-fast",
"gpt-oss-120B (high)": "openai/gpt-oss-120b",
"Grok Code Fast 1": "x-ai/grok-code-fast-1",
"Gemini 2.5 Flash-Lite (Sep)": "google/gemini-2.5-flash-lite",
"Llama Nemotron Super 49B v1.5": "nvidia/llama-3.3-nemotron-super-49b-v1.5",
"gpt-oss-20B (high)": "openai/gpt-oss-20b"
}

In [ ]:
CHINESE_MODELS = []
for model in chinese_models:
    if model in model_lookup:
        print(f"{model}: {model_lookup[model]}")
        CHINESE_MODELS.append(model_lookup[model])
    else:
        print(f"{model}: Not found in lookup")

In [ ]:
AMERICAN_MODELS = []
for model in us_models:
    if model in model_lookup:
        print(f"{model}: {model_lookup[model]}")
        AMERICAN_MODELS.append(model_lookup[model])
    else:
        print(f"{model}: Not found in lookup")

# 3) Config Generation

Generates aligned YAML configurations for each group.
- Language is English for all runs.
- Utility agent: `gemini-2.5-flash`.
- Voting detection mode: `complex`.


In [4]:
# Base paths and groups
BASE_DIR = _REPO_ROOT / 'hypothesis_testing' / 'hypothesis_2'
CONFIGS_BASE = BASE_DIR / 'configs'
LOGS_BASE = BASE_DIR / 'terminal_outputs'
RESULTS_BASE = BASE_DIR / 'results'
TRANSCRIPTS_BASE = BASE_DIR / 'transcripts'

GROUPS = {
    'american': 'American LLMs',
    'chinese': 'Chinese LLMs',
}

# Ensure subfolders exist
for key in GROUPS.keys():
    (CONFIGS_BASE / key).mkdir(parents=True, exist_ok=True)
    (LOGS_BASE / key).mkdir(parents=True, exist_ok=True)
    (RESULTS_BASE / key).mkdir(parents=True, exist_ok=True)
    (TRANSCRIPTS_BASE / key).mkdir(parents=True, exist_ok=True)
CONFIGS_BASE, LOGS_BASE, RESULTS_BASE, TRANSCRIPTS_BASE, GROUPS


(PosixPath('/Users/lucasmuller/Desktop/Githubg/Rawls_v3/hypothesis_testing/hypothesis_2/configs'),
 PosixPath('/Users/lucasmuller/Desktop/Githubg/Rawls_v3/hypothesis_testing/hypothesis_2/terminal_outputs'),
 PosixPath('/Users/lucasmuller/Desktop/Githubg/Rawls_v3/hypothesis_testing/hypothesis_2/results'),
 PosixPath('/Users/lucasmuller/Desktop/Githubg/Rawls_v3/hypothesis_testing/hypothesis_2/transcripts'),
 {'american': 'American LLMs', 'chinese': 'Chinese LLMs'})

In [5]:
# Income class probabilities (must sum to 1.0)
INCOME_CLASS_PROBS = {
    'high': 0.05,
    'medium_high': 0.10,
    'medium': 0.50,
    'medium_low': 0.25,
    'low': 0.10,
}

# American and Chinese model pools (OpenRouter IDs)

def make_agents_with_models(temp: float, models: list[str]) -> list[dict]:
    agents = []
    for i in range(0, 3):  # 4 participant agents
        agents.append({
            'name': f'Agent_{i+1}',
            'personality': 'You are a college student',
            'model': models[i],
            'temperature': float(temp),
            'memory_character_limit': 25000,
            'reasoning_enabled': True,
        })
    return agents

def build_config(temp: float, seed_val: int, models: list[str]) -> dict:
    return {
        'language': 'English',
        'seed': int(seed_val),
        'agents': make_agents_with_models(temp, models),
        'utility_agent_model': 'gemini-2.5-flash',
        'utility_agent_temperature': 0.0,
        'phase2_rounds': 10,
        'distribution_range_phase2': [2, 6],
        'income_class_probabilities': INCOME_CLASS_PROBS,
        'original_values_mode': { 'enabled': True },
        
    }

# Optional: set a global seed for reproducible generation (adjust or comment out)
GLOBAL_SEED = 20000
random.seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)

def _pick_american_models() -> list[str]:
    # Allow repeats; sample each agent independently
    return [random.choice(AMERICAN_MODELS) for _ in range(4)]

def _pick_chinese_models() -> list[str]:
    return [random.choice(CHINESE_MODELS) for _ in range(4)]

def generate_aligned_configs(n: int = 34) -> dict[str, list[Path]]:
    paths: dict[str, list[Path]] = {k: [] for k in GROUPS.keys()}
    for idx in range(1, n + 1):
            temp = random.uniform(0.0, 1.5)
            seed_val = random.randint(0, 2**31 - 1)
            # Build per-group models
            models_american = _pick_american_models()
            models_chinese = _pick_chinese_models()

            # Write configs
            for group_key, models in [('american', models_american), ('chinese', models_chinese)]:
                cfg = build_config(temp=temp, seed_val=seed_val, models=models)
                out_dir = (CONFIGS_BASE / group_key)
                out_dir.mkdir(parents=True, exist_ok=True)
                fname = out_dir / f'hypothesis_2_{group_key}_condition_{idx}_config.yaml'
                with open(fname, 'w') as f:
                    yaml.safe_dump(cfg, f, sort_keys=False)
                paths[group_key].append(fname)
    return paths

# Example (commented):
files_by_group = generate_aligned_configs(n=34)
{k: len(v) for k, v in files_by_group.items()}


NameError: name 'AMERICAN_MODELS' is not defined

# 4) Run Configs 

Select subsets and run with per-group logs/results directories.


In [4]:
def run_group(group_key: str, include_indices=None, include_names=None, concurrency: int = 4, timeout_sec: int | None = None):
    cfg_dir = CONFIGS_BASE / group_key
    logs_dir = LOGS_BASE / group_key
    results_dir = RESULTS_BASE / group_key
    configs = list_config_files(cfg_dir)
    selected = select_configs(configs, include_indices=include_indices, include_names=include_names)
    print(f'[{group_key}] Found {len(configs)} configs; selected {len(selected)}')
    run_results = run_configs_in_parallel(
        selected,
        concurrency=concurrency,
        logs_dir=logs_dir,
        results_dir=results_dir,
        timeout_sec=timeout_sec,
    )
    ok = sum(1 for r in run_results if r.get('ok'))
    print(f'[{group_key}] Completed: {ok}/{len(run_results)} OK')
    return run_results




In [5]:
rr_cn = run_group('chinese', include_indices=list(range(1, 18)) , concurrency=4)


[chinese] Found 34 configs; selected 17
[chinese] Completed: 1/17 OK


In [ ]:
rr_us = run_group('american', include_indices=list(range(1, 18)) , concurrency=2)

## 3) Analysis — Compare Outcomes Across Groups

Build a 5×2 contingency table (rows=principle/disagreement categories; columns=American/Chinese)
and run Fisher–Freeman–Halton exact test via R when available.
Compute Cramér's V with bias correction (as in Hypothesis 1). No Chi-square fallback is included.


In [ ]:
CATEGORIES = [
    'maximizing_floor',
    'maximizing_average',
    'maximizing_average_floor_constraint',
    'maximizing_average_range_constraint',
    'disagreement',
]

def categorize_result(result_path: Path) -> str:
    try:
        with open(result_path, 'r') as f:
            data = json.load(f)
        gi = data.get('general_information', {})
        consensus = gi.get('consensus_reached', False)
        principle = gi.get('consensus_principle')
        if consensus and principle in CATEGORIES:
            return principle
        return 'disagreement'
    except Exception:
        return 'disagreement'

def count_by_group() -> dict[str, Counter]:
    out: dict[str, Counter] = {}
    for k in GROUPS.keys():
        counts = Counter()
        result_files = sorted((RESULTS_BASE / k).glob('*_results.json'))
        for rp in result_files:
            counts[categorize_result(rp)] += 1
        for cat in CATEGORIES:
            counts.setdefault(cat, 0)
        out[k] = counts
    return out

group_counts = count_by_group()
for k, counts in group_counts.items():
    print(f'{k.capitalize()} counts:', dict(counts))

# Build contingency table: rows=categories, cols=[American, Chinese]
col_order = ['american', 'chinese']
contingency = np.vstack([[group_counts[col][cat] for col in col_order] for cat in CATEGORIES])
contingency, CATEGORIES, col_order


American counts: {'maximizing_floor': 0, 'maximizing_average': 0, 'maximizing_average_floor_constraint': 0, 'maximizing_average_range_constraint': 0, 'disagreement': 0}
Chinese counts: {'maximizing_floor': 0, 'maximizing_average': 0, 'maximizing_average_floor_constraint': 0, 'maximizing_average_range_constraint': 0, 'disagreement': 0}


(array([[0, 0],
        [0, 0],
        [0, 0],
        [0, 0],
        [0, 0]]),
 ['maximizing_floor',
  'maximizing_average',
  'maximizing_average_floor_constraint',
  'maximizing_average_range_constraint',
  'disagreement'],
 ['american', 'chinese'])

In [ ]:
def fisher_freeman_halton_pvalue_r(contingency: np.ndarray) -> float | None:
    """Run Fisher–Freeman–Halton test via R's fisher.test if available.
    Returns p-value or None if Rscript not found or fails.
    """
    if shutil.which('Rscript') is None:
        return None
    r_matrix = ','.join(str(int(x)) for x in contingency.flatten(order='C'))
    nrow, ncol = contingency.shape
    r_code = f"""m <- matrix(c({r_matrix}), nrow={nrow}, ncol={ncol}, byrow=TRUE);
f <- tryCatch(fisher.test(m), error=function(e) NA);
if (is.list(f)) cat(f$p.value) else cat('NA')
"""
    import subprocess
    try:
        out = subprocess.check_output(['Rscript', '-e', r_code], stderr=subprocess.STDOUT, text=True)
        out = out.strip()
        return float(out) if out and out != 'NA' else None
    except Exception:
        return None

p_ffh = fisher_freeman_halton_pvalue_r(contingency)
if p_ffh is None:
    print('R not available; skipping Fisher–Freeman–Halton exact test')
else:
    print(f'Fisher–Freeman–Halton exact test p-value: {p_ffh:.6f}')


R not available; skipping Fisher–Freeman–Halton exact test


In [ ]:
def cramers_v(contingency: np.ndarray) -> float:
    from scipy.stats import chi2_contingency
    chi2, _, _, _ = chi2_contingency(contingency)
    n = contingency.sum()
    r, c = contingency.shape
    return float(np.sqrt((chi2 / n) / (min(r - 1, c - 1))))

def bias_corrected_cramers_v(contingency: np.ndarray) -> float:
    from scipy.stats import chi2_contingency
    chi2, _, _, _ = chi2_contingency(contingency)
    n = contingency.sum()
    r, c = contingency.shape
    phi2 = chi2 / n
    r1 = r - 1
    c1 = c - 1
    phi2_corr = max(0.0, phi2 - (r1 * c1) / (n - 1))
    r_corr = r - ((r - 1) ** 2) / (n - 1)
    c_corr = c - ((c - 1) ** 2) / (n - 1)
    denom = min(r_corr - 1, c_corr - 1)
    if denom <= 0:
        return 0.0
    return float(np.sqrt(phi2_corr / denom))

cv = cramers_v(contingency)
cv_corr = bias_corrected_cramers_v(contingency)
print(f"Cramér's V (uncorrected): {cv:.4f}")
print(f"Cramér's V (bias-corrected): {cv_corr:.4f}")


Cramér's V (uncorrected): nan
Cramér's V (bias-corrected): 0.0000


/Users/lucasmuller/Desktop/Githubg/Rawls_v3/.venv/lib/python3.11/site-packages/scipy/stats/contingency.py:135: RuntimeWarning: invalid value encountered in divide
  expected = reduce(np.multiply, margsums) / observed.sum() ** (d - 1)
